# Finetunning LLM LawBot

In [ ]:
!python -V

## Install relevant packages

In [ ]:
%%capture

!pip install unsloth
!pip install bitsandbytes
!pip install unsloth_zoo
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## Import all relevant packages throughout this walkthrough

In [ ]:
# Modules for fine-tuning
from unsloth import FastLanguageModel
import torch # Import PyTorch
from trl import SFTTrainer # Trainer for supervised fine-tuning (SFT)
from unsloth import is_bfloat16_supported # Checks if the hardware supports bfloat16 precision
# Hugging Face modules
from huggingface_hub import login # Lets you login to API
from transformers import TrainingArguments # Defines training hyperparameters
from datasets import load_dataset # Lets you load fine-tuning datasets
# Import weights and biases
import wandb
# Import kaggle secrets
from kaggle_secrets import UserSecretsClient

## Create API keys and login to Hugging Face and Weights and Biases

In [ ]:
user_secrets = UserSecretsClient()
hugging_face_token = user_secrets.get_secret("HF_TOKEN_DEEPSEEK")
wnb_token = user_secrets.get_secret("wnb_token")

login(hugging_face_token)

wandb.login(key=wnb_token)
run = wandb.init(
    project='Fine-tune-DeepSeek-LawBot', 
    job_type="training", 
    anonymous="allow"
)

## Loading Model and the Tokenizer

In [ ]:
max_seq_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=hugging_face_token,
)

## Testing Model on a law use-case before fine-tuning

In [ ]:
prompt_style = """
### Peran:
Ahli Hukum Strategis (Indonesia).

### Struktur Jawaban:
1. Inti Jawaban:
    - Berikan jawaban langsung dan ringkas.
    - Sebutkan implikasi praktis utamanya.
2.  Rincian Analisis:
    a. Isu Pokok: Identifikasi pertanyaan hukum spesifik.
    b. Aturan & Unsur: Uraikan aturan hukum (wajib sitasi pasal/UU TNI) dan unsur-unsurnya.
    c. Penerapan: Analisis logis bagaimana aturan berlaku pada fakta.
3.  Disclaimer.

### Pertanyaan:
{}

### Jawaban:
{}
"""

### Running inference on the model


In [ ]:
question = """apakah tni bisa menjabat menjadi camat"""

FastLanguageModel.for_inference(model)

inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=2048,
    use_cache=True,
)

response = tokenizer.batch_decode(outputs)

print(response[0].split("### Response:")[0])

## Fine-tuning step by step

## Step 1 — Update the system prompt 

In [ ]:
train_prompt_style = """
### Peran:
Ahli Hukum Strategis (Indonesia).

### Struktur Jawaban:
1. Inti Jawaban:
    - Berikan jawaban langsung dan ringkas.
    - Sebutkan implikasi praktis utamanya.
2.  Rincian Analisis:
    a. Isu Pokok: Identifikasi pertanyaan hukum spesifik.
    b. Aturan & Unsur: Uraikan aturan hukum (wajib sitasi pasal/UU TNI) dan unsur-unsurnya.
    c. Penerapan: Analisis logis bagaimana aturan berlaku pada fakta.
3.  Disclaimer.

### Pertanyaan:
{}

### Jawaban:
{}
"""


## Step 2 — Download the fine-tuning dataset and format it for fine-tuning

In [ ]:
import pandas as pd
from datasets import Dataset, concatenate_datasets
df = pd.read_csv("/kaggle/input/dataset-with-format-answer-new/dataset-formated-fix.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
TEST_PROPORTION = 0.1 # 10% dari setiap grup pasal akan menjadi data evaluasi
SEED = 123

train_splits = {}
test_splits = {}

# Dapatkan semua nilai unik dari kolom 'pasal'
unique_pasals = df['pasal'].unique()
print(f"Ditemukan {len(unique_pasals)} pasal unik untuk di-split.")

for pasal_value in unique_pasals:
    # Filter DataFrame untuk pasal saat ini
    group_df = df[df['pasal'] == pasal_value]
    
    # Ubah DataFrame grup menjadi Dataset
    dataset_group = Dataset.from_pandas(group_df)
    
    # Lakukan split HANYA jika grup memiliki lebih dari 1 baris
    if len(dataset_group) > 1:
        split = dataset_group.train_test_split(test_size=TEST_PROPORTION, seed=SEED)
        train_splits[pasal_value] = split['train']
        test_splits[pasal_value] = split['test']
    else:
        # Jika grup terlalu kecil, masukkan semuanya ke training set
        train_splits[pasal_value] = dataset_group
        # Buat test set kosong untuk pasal ini
        test_splits[pasal_value] = None 
        print(f"Peringatan: Grup untuk pasal '{pasal_value}' hanya memiliki 1 item, dimasukkan semua ke training set.")

print("\n--- Ringkasan Hasil Split per Pasal ---")
for pasal_value in unique_pasals:
    train_count = len(train_splits[pasal_value])
    test_count = len(test_splits[pasal_value]) if test_splits[pasal_value] else 0
    print(f"Pasal {pasal_value}: Train = {train_count} | Test = {test_count}")

In [ ]:
final_train_list = [ds for ds in train_splits.values() if ds is not None]
final_test_list = [ds for ds in test_splits.values() if ds is not None]

# Gabungkan dataset
final_train_dataset_raw = concatenate_datasets(final_train_list).shuffle(seed=SEED)
final_eval_dataset_raw = concatenate_datasets(final_test_list).shuffle(seed=SEED)

print("\n--- Informasi Dataset Gabungan ---")
print(f"Total data Latihan (Train): {len(final_train_dataset_raw)}")
print(f"Total data Evaluasi (Test/Eval): {len(final_eval_dataset_raw)}")
print("-" * 50)

In [ ]:
train_prompt_style = "### Pertanyaan:\n{0}\n\n### Jawaban:\n{1}"
EOS_TOKEN = tokenizer.eos_token  
print(f"EOS_TOKEN: {EOS_TOKEN}")

def formatting_prompts_func(examples):
    inputs = examples["Question"]
    outputs = examples["Answer"]
    texts = []
    for input_text, output_text in zip(inputs, outputs):
        text = train_prompt_style.format(input_text, output_text) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Terapkan fungsi map
train_dataset = final_train_dataset_raw.map(formatting_prompts_func, batched=True, remove_columns=final_train_dataset_raw.column_names)
eval_dataset = final_eval_dataset_raw.map(formatting_prompts_func, batched=True, remove_columns=final_eval_dataset_raw.column_names)

print("\n--- Preview Hasil Akhir ---")
print("Dataset Latihan setelah di-format:")
print(train_dataset)
print("\nDataset Evaluasi setelah di-format:")
print(eval_dataset)

# Contoh preview hasil formatting
print("\nContoh teks dari dataset latihan yang telah diformat:")
print(train_dataset["text"][0])

## Step 3 — Setting up the model using LoRA

In [ ]:
model_lora_lawbot = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "v_proj", "k_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
import os
from transformers import TrainerCallback, TrainingArguments, TrainerState, TrainerControl

# --- DEFINISIKAN CALLBACK KUSTOM DI SINI ---

class SavePeftModelCallback(TrainerCallback):
    """
    Callback kustom untuk menyimpan model adapter LoRA (PEFT)
    di setiap akhir epoch.
    """
    def __init__(self, tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        
    def on_epoch_end(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        **kwargs,
    ):
        # Dapatkan nomor epoch saat ini (state.epoch adalah float, misal 1.0, 2.0)
        epoch_num = int(state.epoch)
        
        # Tentukan path untuk menyimpan model epoch ini
        save_path = os.path.join(args.output_dir, f"epoch-{epoch_num}")
        
        # Ambil model dari kwargs yang disediakan oleh Trainer
        model = kwargs["model"]
        
        # Simpan adapter LoRA
        model.save_pretrained(save_path)
        
        # Ambil tokenizer juga dan simpan
        self.tokenizer.save_pretrained(save_path)
        
        print(f"Model adapter LoRA untuk epoch {epoch_num} telah disimpan di: {save_path}")
        return control


# --- MODIFIKASI SFTTrainer ANDA ---

trainer_lawbot = SFTTrainer(
    model=model_lora_lawbot,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    callbacks=[SavePeftModelCallback(tokenizer=tokenizer)],
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=16,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="no"
    ),
)

## Step 4 — Model training! 

In [ ]:
# Start the fine-tuning process
trainer_stats = trainer_lawbot.train()

In [ ]:
import matplotlib.pyplot as plt
log_history = trainer_lawbot.state.log_history
train_loss, eval_loss, steps = [], [], []

for log in log_history:
    if "loss" in log:
        train_loss.append(log["loss"])
        steps.append(log["step"])
    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])

# Plot
plt.plot(steps[:len(train_loss)], train_loss, label="Training Loss")
plt.plot(steps[-len(eval_loss):], eval_loss, label="Validation Loss")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# wandb.finish()

## Step 5 — Run model inference after fine-tuning

In [ ]:
question = """Apa arti dari “berada di bawah Presiden” dalam konteks TNI?"""

FastLanguageModel.for_inference(model_lora_lawbot)

inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model_lora_lawbot.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=2048,
    use_cache=True,
)

response = tokenizer.batch_decode(outputs)

print(response[0].split("### Jawaban:")[1])

## Saving Model

In [ ]:
from IPython.display import FileLink

# Setelah training selesai
trainer_lawbot.save_model("model_lora_lawbot_final")
tokenizer.save_pretrained("model_lora_lawbot_final")

!zip -r model_lora_lawbot.zip model_lora_lawbot

# # Membuat link untuk download
# FileLink("model_lora_lawbot.zip")

## Evaluasi

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel
from transformers import AutoTokenizer

import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

# Path ke folder model yang sudah disimpan
model_dir = "/kaggle/input/llama3.1-adapter/pytorch/default/1/model_lora_lawbot_final"

# 3. Load LoRA weights yang sudah difinetune
model_lora_lawbot = PeftModel.from_pretrained(model, model_dir)

In [ ]:
# Sampling untuk debug
num_samples = 2
SEED = 42

# Ambil 5 sampel acak dari dataset evaluasi mentah
# Cara yang benar untuk sampling di `datasets` adalah shuffle() lalu select()
print(f"Mengambil {num_samples} sampel acak dari dataset evaluasi...")
if len(final_eval_dataset_raw) > num_samples:
    sampling_dataset = final_eval_dataset_raw.shuffle(seed=SEED).select(range(num_samples))
else:
    print(f"Peringatan: Ukuran eval_dataset_raw ({len(final_eval_dataset_raw)}) lebih kecil dari jumlah sampel ({num_samples}). Menggunakan seluruh dataset.")
    sampling_dataset = final_eval_dataset_raw

print(f"Dataset sampel yang akan dievaluasi berisi: {len(sampling_dataset)} data.")
print("-" * 50)

In [ ]:
import pandas as pd
import subprocess
import sys
from tqdm.auto import tqdm
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

model_lora_lawbot.to(device)
FastLanguageModel.for_inference(model_lora_lawbot) # Optimisasi Unsloth

# --- 2. Proses Evaluasi ---

# Inisialisasi list untuk menyimpan hasil
predictions = []
references  = []
questions= []

# Iterasi melalui dataset evaluasi
for ex in tqdm(final_eval_dataset_raw, desc="Mengevaluasi Model"):
# for ex in tqdm(sampling_dataset, desc="Mengevaluasi Model"):
    question = ex["Question"]
    reference_answer = ex["Answer"]
    
    inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

    outputs = model_lora_lawbot.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=1024,
        use_cache=True,
        
    )
    response = tokenizer.batch_decode(outputs)
    generated_answer = response[0].split("### Jawaban:")[1].strip().split("<|end_of_text|>")[0].strip()

        
    # Tambahkan hasil ke list
    predictions.append(generated_answer)
    references.append(reference_answer)
    questions.append(question)

In [ ]:
df_results = pd.DataFrame({
    'Questions': questions,
    'Reference Answer': references,
    'Generate Answer': predictions
})
df_results.to_csv('result_generate.csv',index=False)
df_results

In [ ]:
print(f"Pertanyaan : {df_results['Questions'][0]} \n")
print(f"Ground Truth : {df_results['Reference Answer'][0]}\n")
print(f"Generate Answer : {df_results['Generate Answer'][0]}\n")

In [ ]:
import pandas as pd
import subprocess
import sys

# --- 1. Setup Library ---

# Pastikan library 'evaluate' dan 'rouge_score' terinstal
try:
    import evaluate
except ImportError:
    print("Menginstal library 'evaluate' dan 'rouge_score'...")
    subprocess.run([sys.executable, "-m", "pip", "install", "evaluate", "rouge_score"], check=True)
    import evaluate

# --- 2. Muat Data dari File CSV ---

# Definisikan path ke file hasil generate Anda
file_path = '/kaggle/working/result_generate.csv'

try:
    # Baca file CSV ke dalam DataFrame
    df_eval = pd.read_csv(file_path)
    
except FileNotFoundError:
    print(f"Error: File '{file_path}' tidak ditemukan. Pastikan file sudah dibuat dari langkah sebelumnya.")
    # Keluar dari skrip jika file tidak ada
    sys.exit()

# --- 3. Siapkan Data untuk Evaluasi ROUGE ---

try:
    predictions_from_csv = df_eval['Generate Answer'].tolist()
    references_from_csv = df_eval['Reference Answer'].tolist()
    
    print(f"Data siap untuk evaluasi. Jumlah prediksi: {len(predictions_from_csv)}, Jumlah referensi: {len(references_from_csv)}")
    
except KeyError as e:
    print(f"Error: Kolom {e} tidak ditemukan di file CSV.")
    print("Pastikan file 'result_generate.csv' Anda memiliki kolom 'Generate Answer' dan 'Reference Answer'.")
    print(f"Kolom yang tersedia di file Anda: {df_eval.columns.tolist()}")
    sys.exit()

# --- 4. Kalkulasi dan Tampilkan Hasil ROUGE dari CSV ---

# Muat metrik ROUGE
rouge = evaluate.load("rouge")

print("\nMengkalkulasi skor ROUGE dari data file CSV...")
rouge_scores = rouge.compute(
    predictions=predictions_from_csv, 
    references=references_from_csv
)

print("\n--- Hasil Akhir Evaluasi ROUGE ---")
# Mengalikan dengan 100 untuk persentase dan memformat 2 angka di belakang koma
print(f"ROUGE-1 (Precision & Recall): {rouge_scores['rouge1'] * 100:.2f}")
print(f"ROUGE-2 (Bigram Overlap):    {rouge_scores['rouge2'] * 100:.2f}")
print(f"ROUGE-L (Longest Subsequence): {rouge_scores['rougeL'] * 100:.2f}")
print(f"ROUGE-Lsum (Summary Level):    {rouge_scores['rougeLsum'] * 100:.2f}")
print("-" * 40)

In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm
import numpy as np
import subprocess
import sys

device = "cuda" if torch.cuda.is_available() else "cpu"
df_results = pd.read_csv(file_path)

# --- 1. Instalasi dan Setup Library Tambahan ---

# Pastikan library yang dibutuhkan untuk evaluasi semantik terinstal
print("Memeriksa dan menginstal library yang dibutuhkan...")
try:
    import nltk
    from sentence_transformers import SentenceTransformer, util
    from sklearn.metrics.pairwise import cosine_similarity
except ImportError:
    print("Menginstal 'sentence-transformers', 'nltk', dan 'scikit-learn'...")
    subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers", "nltk", "scikit-learn"], check=True)
    from sentence_transformers import SentenceTransformer, util
    from sklearn.metrics.pairwise import cosine_similarity
    import nltk

# Download tokenizer 'punkt' dari NLTK untuk memecah kalimat
try:
    nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError:
    print("Mengunduh NLTK 'punkt' tokenizer...")
    nltk.download('punkt')

# --- 2. Muat Model Embedding ---

# Memuat model embedding multilingual yang kuat.
# Qwen/Qwen2-57B-A14B-instruct-embedding adalah model yang sangat besar. 
# Untuk efisiensi, kita gunakan model yang lebih ringan namun tetap powerful untuk multilingual.
# 'paraphrase-multilingual-mpnet-base-v2' adalah pilihan yang sangat baik.
embedding_model_name = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
print(f"Memuat model embedding ({embedding_model_name})...")
embedding_model = SentenceTransformer(embedding_model_name, device=device)
print("Model embedding berhasil dimuat.")
print("-" * 50)


# --- 3. Fungsi untuk Menghitung Skor Semantik ---

def calculate_semantic_scores(reference, prediction, model, threshold=0.85):
    """
    Menghitung precision, recall, dan F1-score berdasarkan kesamaan semantik.
    """
    # Memecah jawaban menjadi kalimat-kalimat
    ref_sentences = nltk.sent_tokenize(reference)
    pred_sentences = nltk.sent_tokenize(prediction)

    if not pred_sentences or not ref_sentences:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

    # Generate embeddings untuk setiap kalimat
    ref_embeddings = model.encode(ref_sentences, convert_to_tensor=True)
    pred_embeddings = model.encode(pred_sentences, convert_to_tensor=True)

    # Hitung cosine similarity
    cosine_scores = util.cos_sim(pred_embeddings, ref_embeddings)

    # Hitung Precision
    pred_matches = 0
    for i in range(len(pred_sentences)):
        if cosine_scores.shape[1] > 0 and torch.max(cosine_scores[i]) > threshold:
            pred_matches += 1
    precision = pred_matches / len(pred_sentences)

    # Hitung Recall
    ref_matches = 0
    for j in range(len(ref_sentences)):
        if cosine_scores.shape[0] > 0 and torch.max(cosine_scores[:, j]) > threshold:
            ref_matches += 1
    recall = ref_matches / len(ref_sentences)
    
    # Hitung F1 Score
    if (precision + recall) == 0:
        f1 = 0.0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)
        
    return {'precision': float(precision), 'recall': float(recall), 'f1': float(f1)}


# --- 4. Terapkan Fungsi Evaluasi ke DataFrame Hasil ---

# Asumsikan df_results sudah ada dari kode sebelumnya dan memiliki kolom
# 'Reference Answer' dan 'Generate Answer'.
# Contoh:
# df_results = pd.read_csv('result_generate.csv')

scores = []
# Iterasi melalui setiap baris di DataFrame hasil
for index, row in tqdm(df_results.iterrows(), total=df_results.shape[0], desc="Menghitung Skor Semantik"):
    
    # --- MODIFIKASI NAMA KOLOM DI SINI ---
    # Menggunakan nama kolom 'Reference Answer' dan 'Generate Answer'
    # Menggunakan str() untuk memastikan tipe data string dan menangani NaN
    ref = str(row['Reference Answer'])
    pred = str(row['Generate Answer'])
    
    # Hitung skor untuk baris saat ini
    score = calculate_semantic_scores(ref, pred, embedding_model)
    scores.append(score)

# Tambahkan skor ke DataFrame
df_scores = pd.DataFrame(scores)
df_results_with_scores = pd.concat([df_results, df_scores], axis=1)


# --- 5. Tampilkan Hasil Akhir ---

# Hitung rata-rata skor
avg_precision = df_results_with_scores['precision'].mean()
avg_recall = df_results_with_scores['recall'].mean()
avg_f1 = df_results_with_scores['f1'].mean()

print("\n--- Hasil Evaluasi Semantik (Rata-rata) ---")
print(f"Average Precision : {avg_precision * 100:.2f}%")
print(f"Average Recall    : {avg_recall * 100:.2f}%")
print(f"Average F1-Score  : {avg_f1 * 100:.2f}%")
print("-" * 50)


# Simpan hasil akhir yang sudah berisi skor semantik
output_scored_csv_path = 'result_generate_with_scores.csv'
df_results_with_scores.to_csv(output_scored_csv_path, index=False)
print(f"\nHasil evaluasi lengkap dengan skor berhasil disimpan ke: {output_scored_csv_path}")